# Notebook 03: CLIP Zero-Shot sobre prendas

**Propósito:** evaluar un CLIP pre-entrenado como clasificador zero-shot de prendas (historias bccv + bcdj): 30 imágenes etiquetadas a mano, prompts estructurados, accuracy top-1/top-3 y doc de prompts. Además deriva 10K+ pares (imagen, texto) + splits para fine-tuning futuro.

**Objetos que define:** `catalogo`, `muestra_30`, `labels_df`, `acc_top1`, `acc_top3`, `pares_df`, `fitb_no` (no aplica: FITB vive en el 02)

**Objetos que define (nombres finales):** `catalogo`, `muestra_30`, `labels_df`, `acc_top1`, `acc_top3`, `pares_df`

## 1. Entorno y GPU

Detecta Colab vs local. CLIP necesita GPU para correr comodo (Colab T4); en CPU funciona pero lento.

In [ ]:
import sys, platform
from pathlib import Path

# Deteccion de entorno (mismo patron que 01/02)
IS_COLAB = Path("/content").exists()
REPO_ROOT = Path("/content/MACHINE-BULLS") if IS_COLAB and Path("/content/MACHINE-BULLS").exists() else (Path("/content") if IS_COLAB else Path.cwd().resolve())
if not IS_COLAB:
    for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
        if (parent / "notebooks" / "03_clip_zero_shot.ipynb").exists():
            REPO_ROOT = parent
            break
print(f"Entorno: {'Colab' if IS_COLAB else 'Local'} | Repo root: {REPO_ROOT} | Python: {sys.version.split()[0]}")
print(f"Plataforma: {platform.platform()}")

try:
    import torch
    print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("[WARN] Sin GPU: CLIP corre en CPU (lento pero funciona para 30 imagenes).")
except ImportError:
    print("[WARN] PyTorch no instalado (ver Seccion 1b).")


## 1b. Dependencias y MLflow

En Colab correr instalacion + reiniciar runtime si cambia protobuf. En local usar el venv del backend o pip del sistema.

In [ ]:
# Colab: descomentar y correr una vez, luego reiniciar runtime si pide protobuf/numpy.
# %pip install -q "mlflow==3.16.1" transformers scikit-learn pillow huggingface_hub matplotlib
print("[INFO] Si es sesion fresca de Colab: descomenta el %pip de arriba, instala y reinicia runtime.")


In [ ]:
import mlflow
from pathlib import Path

# Tracking automatico por entorno (patron 01/02): ngrok en Colab, localhost en local.
IS_COLAB = globals().get("IS_COLAB", Path("/content").exists())
REMOTE_URI = ("https://humorous-trusting-domelike.ngrok-free.dev"
              if IS_COLAB else "http://localhost:5000")
try:
    mlflow.set_tracking_uri(REMOTE_URI)
    mlflow.set_experiment("03-clip-zero-shot")
    print(f"[OK] MLflow ({'remoto' if IS_COLAB else 'local'}): {REMOTE_URI}")
except Exception as e:
    print(f"[WARN] MLflow no levantado ({e})")
    if IS_COLAB:
        print("   Levanta el tunel: uv run python backend/scripts/mlflow_ngrok.py")
    else:
        print("   Levanta local: uv run bash backend/scripts/mlflow_ui.sh")
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(10)


## 2. Carga autocontenida del catalogo (no depende del 01/02)

Los mismos 4 JSON de metadatos via `hf_hub_download`. Requiere login HF previo (`huggingface_hub.login()`) si el dataset es gated.

In [ ]:
from huggingface_hub import hf_hub_download, login
import json

try:
    login()
except Exception as e:
    print(f"[WARN] login HF fallo o ya activo: {e}")

REPO_ID = "ArtmeScienceLab/Garments2Look"
REPO_TYPE = "dataset"
OUTFIT_METADATA_FILES = ["mytheresa_outfit_v1.0_2512.json", "polyvore_outfit_v1.0_2512.json"]
IMAGE_METADATA_FILES = ["mytheresa_image_v1.0_2512.json", "polyvore_image_v1.0_2512.json"]

catalogo = {}
for fname in IMAGE_METADATA_FILES:
    path = hf_hub_download(repo_id=REPO_ID, filename=fname, repo_type=REPO_TYPE)
    with open(path, "r", encoding="utf-8") as f:
        catalogo.update(json.load(f))
print(f"Prendas en catalogo: {len(catalogo)}")
assert len(catalogo) > 100000, "catalogo incompleto, revisa la descarga"


## 3. Muestra de 30 imagenes por streaming + plantilla de etiquetado

Baja 30 fotos de `polyvore/images.tar.gz` sin persistir el tar. Genera `sample_data/clip_labels_30.csv` con categoria sugerida: **completa la columna `manual_label` a mano** (30 filas, minutos) antes de seguir. Sin etiquetas manuales la celda 5 falla en voz alta.

In [ ]:
import csv
import requests
import tarfile
from pathlib import Path
from PIL import Image

SAMPLE_DIR = REPO_ROOT / "sample_data"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
LABELS_CSV = SAMPLE_DIR / "clip_labels_30.csv"

muestra_30 = []  # (garment_id, PIL.Image)
url = f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/polyvore/images.tar.gz"
print(f"Conectando al stream de: {url}")
response = requests.get(url, stream=True)
try:
    response.raise_for_status()
    with tarfile.open(fileobj=response.raw, mode="r|*") as tar:
        for member in tar:
            if member.isreg() and member.name.lower().endswith((".png", ".jpg", ".jpeg")):
                f = tar.extractfile(member)
                if f is None:
                    continue
                import os
                gid = os.path.splitext(os.path.basename(member.name))[0]
                img = Image.open(f).convert("RGB")
                img.thumbnail((384, 384))
                muestra_30.append((gid, img))
                if len(muestra_30) >= 30:
                    break
finally:
    response.close()  # sin esto el tar sigue bajando en fondo
print(f"[OK] Muestra: {len(muestra_30)} imagenes")
assert len(muestra_30) == 30, "muestra incompleta, reintenta el streaming"

# Plantilla de etiquetado manual (categoria sugerida desde el catalogo)
def _sugerida(gid):
    meta = catalogo.get(gid, {})
    t = (meta.get("type") or "desconocido").split("::")
    return t[0]

if not LABELS_CSV.exists():
    with LABELS_CSV.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["image_id", "sugerida", "manual_label"])
        for gid, _ in muestra_30:
            w.writerow([gid, _sugerida(gid), ""])
    print(f"[OK] Plantilla creada: {LABELS_CSV}")
else:
    print(f"[OK] Plantilla existente: {LABELS_CSV} (no se sobrescribe)")
print("COMPLETA la columna manual_label (30 filas) y sigue a la celda 5.")


## 4. Checkpoint de etiquetas manuales

Falla en voz alta si falta alguna etiqueta. No hay fallback sintetico: la accuracy se mide contra labels humanos.

In [ ]:
import csv
import pandas as pd

labels_df = pd.read_csv(LABELS_CSV, dtype=str).fillna("")
assert list(labels_df.columns) == ["image_id", "sugerida", "manual_label"], "esquema del CSV"
faltantes = labels_df[labels_df["manual_label"].str.strip() == ""]
assert len(faltantes) == 0, f"faltan {len(faltantes)} etiquetas manuales en {LABELS_CSV}"
assert len(labels_df) == 30, "se esperan 30 filas etiquetadas"
print(f"[OK] 30/30 etiquetas manuales. Distribucion: {dict(labels_df['manual_label'].value_counts())}")


## 5. Eval zero-shot CLIP

Modelo `openai/clip-vit-base-patch32`, prompts estructurados por categoria. Top-1 y top-3 contra labels manuales (minimo 30, AC bccv).

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[OK] CLIP en device: {DEVICE}")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("[OK] CLIP cargado")

# Candidatos = categorias presentes en la muestra (sin_catalogo afuera)
CANDIDATAS = sorted({c for c in labels_df["manual_label"].unique() if c != "sin_catalogo"})
assert len(CANDIDATAS) >= 2, "se necesitan >=2 categorias para rankear"
PROMPTS = [f"a photo of {c} clothing item" for c in CANDIDATAS]
print(f"Categorias candidatas ({len(CANDIDATAS)}): {CANDIDATAS}")

import torch as _t
top1_ok = top3_ok = evaluadas = 0
detalle = []
for gid, img in muestra_30:
    verdad = labels_df.loc[labels_df["image_id"] == gid, "manual_label"].values
    if len(verdad) == 0 or verdad[0] == "sin_catalogo":
        continue
    inputs = clip_proc(text=PROMPTS, images=img, return_tensors="pt", padding=True).to(DEVICE)
    with _t.no_grad():
        logits = clip_model(**inputs).logits_per_image.softmax(dim=1)[0]
    orden = logits.argsort(descending=True)
    pred1 = CANDIDATAS[int(orden[0])]
    top3 = {CANDIDATAS[int(k)] for k in orden[:3]}
    top1_ok += pred1 == verdad[0]
    top3_ok += verdad[0] in top3
    evaluadas += 1
    detalle.append((gid, verdad[0], pred1))
assert evaluadas >= 30 or len(muestra_30) - evaluadas <= 5, "muy pocas evaluables"
acc_top1 = top1_ok / evaluadas
acc_top3 = top3_ok / evaluadas
print(f"[OK] Evaluadas: {evaluadas} | top-1: {acc_top1:.3f} | top-3: {acc_top3:.3f}")


## 6. Doc de prompts efectivos (`docs/ml/clip_prompts.md`)

Genera el archivo pedido por el AC bccv con los prompts usados y el resultado. En Colab queda en la sesion (copiar al repo); en local se escribe directo.

In [ ]:
prompts_doc = REPO_ROOT / "docs" / "ml" / "clip_prompts.md"
prompts_doc.parent.mkdir(parents=True, exist_ok=True)
cuerpo = (
    "# CLIP prompts efectivos (notebook 03)\n\n"
    f"Modelo: openai/clip-vit-base-patch32 | top-1: {acc_top1:.3f} | top-3: {acc_top3:.3f} | n={evaluadas}\n\n"
    "## Template\n\n```\na photo of {categoria} clothing item\n```\n\n"
    "## Categorias candidatas\n\n" + "".join(f"- {c}\n" for c in CANDIDATAS) + "\n"
    "## Nota\n\nPrompts en ingles rinden mejor que traducciones literales en este checkpoint.\n"
)
prompts_doc.write_text(cuerpo, encoding="utf-8")
print(f"[OK] Doc escrito: {prompts_doc}")


## 7. Pares imagen-texto 10K+ y splits (AC bcdj)

Deriva pares del catalogo (sin descargar imagenes): texto = nombre + color + tipo. Shuffle con seed 42, split 80/10/10, conteos validados. Manifiesto JSONL en `sample_data/` (liviano, commiteable si <5MB).

In [ ]:
import json
import random

random.seed(42)
pares = []
for gid, meta in catalogo.items():
    nombre = (meta.get("name") or "").strip()
    tipo = (meta.get("type") or "").strip()
    color = (meta.get("color") or "").strip()
    if not nombre or not tipo:
        continue
    texto = f"{nombre}, {color} {tipo}" if color else f"{nombre}, {tipo}"
    pares.append({"garment_id": gid, "text": texto})
random.shuffle(pares)
pares = pares[:12000]
n = len(pares)
assert n >= 10000, f"solo {n} pares de calidad, insuficientes para el AC"
n_train = int(n * 0.8)
n_val = int(n * 0.1)
splits = {"train": pares[:n_train], "val": pares[n_train:n_train + n_val], "test": pares[n_train + n_val:]}
for k, v in splits.items():
    assert len(v) > 0, f"split {k} vacio"
    p = SAMPLE_DIR / f"clip_pairs_{k}.jsonl"
    with p.open("w", encoding="utf-8") as fh:
        for row in v:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"[OK] {k}: {len(v)} pares -> {p.name} ({p.stat().st_size / 1024:.0f} KB)")
pares_df = {k: len(v) for k, v in splits.items()}
print(f"[OK] Total: {n} pares (train/val/test = {pares_df['train']}/{pares_df['val']}/{pares_df['test']})")


## 8. Log a MLflow

Parametros, metricas y artefactos del experimento `03-clip-zero-shot`.

In [ ]:
import mlflow
import tempfile

with mlflow.start_run(run_name="clip-zero-shot") as run:
    mlflow.log_param("modelo", "openai/clip-vit-base-patch32")
    mlflow.log_param("n_muestra", 30)
    mlflow.log_param("device", DEVICE)
    mlflow.log_metric("acc_top1", float(acc_top1))
    mlflow.log_metric("acc_top3", float(acc_top3))
    mlflow.log_metric("n_pares", float(sum(pares_df.values())))
    mlflow.set_tag("phase", "clip-eval")
    mlflow.set_tag("notebook", "03_clip")
    with tempfile.TemporaryDirectory() as tmp:
        import shutil
        shutil.copy(str(LABELS_CSV), f"{tmp}/clip_labels_30.csv")
        mlflow.log_artifact(f"{tmp}/clip_labels_30.csv", artifact_path="clip")
        mlflow.log_artifact(str(prompts_doc), artifact_path="clip")
    print(f"[OK] MLflow run 03: {run.info.run_id}")


## 9. Veredicto y cierre

CLIP zero-shot medido contra 30 labels humanos + 10K pares listos para fine-tuning. Cierra bccv (eval + report + prompts) y bcdj (pares + splits). Siguiente: skeleton de training (bccr, ya en backend) consume estos splits.